# 🚀 ClarityCareers — Improved Model Training

This notebook trains **two improved models** on top of the existing pipeline without touching the current `advanced-model`.

---

## 🔧 What's improved over the baseline `advanced-model` (61.67% accuracy)

| Option | Change | Expected Gain |
|--------|--------|---------------|
| **Option 1** | `MultipleNegativesRankingLoss` instead of `CosineSimilarityLoss` | +5–8% |
| **Option 3** | Cross-Encoder architecture (reads both texts together) | +10–15% |
| **Option 4** | Threshold optimized for **F1-score** instead of accuracy | +1–3% |

---

## 📦 Output Models
After training you will download:
- **`improved-biencoder.zip`** → unzip and place at `models/improved-biencoder/` in the project
- **`crossencoder.zip`** → unzip and place at `models/crossencoder/` in the project

The current `models/advanced-model/` will **not be touched**.

---

## ▶️ How to run
1. Go to **Runtime → Change runtime type → GPU** (T4 recommended)
2. Run **Cell 2** to install packages
3. Run **Cell 3** to upload your dataset (`job_applicant_dataset.csv`)
4. Run all remaining cells in order

In [ ]:
# ============================================================
# CELL 2 — Install Dependencies
# ============================================================
!pip install -q sentence-transformers scikit-learn pandas numpy tqdm scipy
!python -m spacy download en_core_web_sm -q
print('✅ All packages installed')

In [ ]:
# ============================================================
# CELL 3 — Upload Dataset
# ============================================================
# Choose how to load your dataset:
#   'upload' → upload directly from your computer (default)
#   'drive'  → load from Google Drive

UPLOAD_METHOD = 'upload'   # ← change to 'drive' if needed

if UPLOAD_METHOD == 'upload':
    from google.colab import files
    print('📤 Please upload your job_applicant_dataset.csv file:')
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

elif UPLOAD_METHOD == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    # ↓ Update this path to match your Drive location
    DATASET_PATH = '/content/drive/MyDrive/job_applicant_dataset.csv'

print(f'✅ Dataset ready: {DATASET_PATH}')

---
## ⚙️ Imports & Configuration

In [ ]:
# ============================================================
# CELL 4 — Imports
# ============================================================
import pandas as pd
import numpy as np
import re, os, json, shutil
from datetime import datetime
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, f1_score
)

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader
import torch
from scipy.special import expit  # sigmoid

import spacy
nlp = spacy.load('en_core_web_sm')

print('✅ All imports successful')
print(f'🔧 PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'🖥️  GPU: ✅ {torch.cuda.get_device_name(0)}')
else:
    print('🖥️  GPU: ❌ CPU only (training will be slower — switch to GPU runtime!)')

In [ ]:
# ============================================================
# CELL 5 — Configuration
# ============================================================
CONFIG = {
    # ── Data ──────────────────────────────────────────────
    'random_seed':            42,
    'sample_size':            12000,   # max rows to use from CSV
    'use_data_augmentation':  True,
    'use_hard_negatives':     True,

    # ── Bi-Encoder (Option 1 + 4) ─────────────────────────
    'biencoder_base':         'sentence-transformers/all-MiniLM-L6-v2',
    'biencoder_output':       'models/improved-biencoder',
    'biencoder_batch_size':   32,
    'biencoder_epochs':       10,
    'biencoder_lr':           2e-5,
    'biencoder_warmup':       100,
    'biencoder_max_seq':      256,
    'skill_weight':           0.35,   # weight for skill-overlap hybrid score

    # ── Cross-Encoder (Option 3 + 4) ──────────────────────
    'crossencoder_base':      'cross-encoder/ms-marco-MiniLM-L-6-v2',
    'crossencoder_output':    'models/crossencoder',
    'crossencoder_batch_size': 16,
    'crossencoder_epochs':    3,
    'crossencoder_lr':        2e-5,
    'crossencoder_max_len':   512,
}

os.makedirs(CONFIG['biencoder_output'], exist_ok=True)
os.makedirs(CONFIG['crossencoder_output'], exist_ok=True)
np.random.seed(CONFIG['random_seed'])

print('✅ Configuration ready')
print(f"   Bi-encoder output  → {CONFIG['biencoder_output']}")
print(f"   Cross-encoder output → {CONFIG['crossencoder_output']}")

---
## 🔧 Helper Functions

In [ ]:
# ============================================================
# CELL 6 — Helper Functions (same as original pipeline)
# ============================================================

def extract_skills(text):
    """Extract technical skills from text using regex patterns."""
    skill_patterns = [
        r'\b(python|java|javascript|c\+\+|sql|r\b|scala|ruby|php)\b',
        r'\b(machine learning|deep learning|nlp|data science|ai|ml)\b',
        r'\b(tensorflow|pytorch|keras|scikit-learn|pandas|numpy)\b',
        r'\b(aws|azure|gcp|docker|kubernetes|jenkins)\b',
        r'\b(react|angular|vue|node|django|flask|spring)\b',
        r'\b(mysql|postgresql|mongodb|redis|oracle)\b',
        r'\b(git|agile|scrum|devops|ci/cd)\b',
    ]
    skills = set()
    text_lower = text.lower()
    for pattern in skill_patterns:
        skills.update(re.findall(pattern, text_lower))
    return list(skills)


def skill_overlap_score(resume, jd):
    """Calculate fraction of JD skills that appear in the resume."""
    rs = set(extract_skills(resume))
    js = set(extract_skills(jd))
    if not js:
        return 0.0
    return len(rs & js) / len(js)


def augment_text(text, n=1):
    """Simple synonym-replacement augmentation."""
    synonyms = {
        'experience': ['expertise', 'background', 'knowledge'],
        'develop':    ['create', 'build', 'design'],
        'manage':     ['oversee', 'coordinate', 'supervise'],
        'work':       ['collaborate', 'operate', 'function'],
        'team':       ['group', 'unit', 'department'],
    }
    variations = [text]
    for _ in range(n):
        new = text
        for word, syns in synonyms.items():
            if word in text.lower():
                new = new.replace(word, np.random.choice(syns))
        if new != text:
            variations.append(new)
    return variations


print('✅ Helper functions defined')

---
## 📊 Load & Prepare Dataset

In [ ]:
# ============================================================
# CELL 7 — Load Data
# ============================================================
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=['Resume', 'Job Description', 'Best Match'])
df = df[df['Resume'].str.len() > 50]
df = df[df['Job Description'].str.len() > 50]

if len(df) > CONFIG['sample_size']:
    df = df.sample(CONFIG['sample_size'], random_state=CONFIG['random_seed'])

print(f'✅ Loaded {len(df):,} records after cleaning')

class_counts = df['Best Match'].value_counts()
print(f'\n📊 Class Distribution:')
print(f"   Matched (1):     {class_counts.get(1, 0):,} ({class_counts.get(1,0)/len(df)*100:.1f}%)")
print(f"   Not Matched (0): {class_counts.get(0, 0):,} ({class_counts.get(0,0)/len(df)*100:.1f}%)")

In [ ]:
# ============================================================
# CELL 8 — Skill Features + Data Augmentation
# ============================================================
print('🔧 Computing skill overlap features...')
df['skill_overlap'] = [
    skill_overlap_score(r, j)
    for r, j in tqdm(zip(df['Resume'], df['Job Description']), total=len(df))
]

if CONFIG['use_data_augmentation']:
    print('\n🔄 Augmenting minority class...')
    minority_class = class_counts.idxmin()
    minority_df = df[df['Best Match'] == minority_class]
    augmented = []
    for _, row in tqdm(minority_df.head(1000).iterrows(), total=min(1000, len(minority_df))):
        for var in augment_text(row['Resume'], 1)[1:]:
            new_row = row.copy()
            new_row['Resume'] = var
            augmented.append(new_row)
    if augmented:
        df = pd.concat([df, pd.DataFrame(augmented)], ignore_index=True)
    print(f'✅ After augmentation: {len(df):,} records')

# Train / Val / Test split (same ratios as original pipeline: 80/10/10)
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=CONFIG['random_seed'], stratify=df['Best Match']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=CONFIG['random_seed'], stratify=temp_df['Best Match']
)

print(f'\n📦 Final Split:')
print(f'   Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')

---
## 🧠 Improved Bi-Encoder Training
**Changes from the original `advanced-model`:**
- ✅ **Option 1** — `MultipleNegativesRankingLoss` replaces `CosineSimilarityLoss`
  - Training uses positive pairs + hard-negative triplets
  - Every other item in the batch acts as an implicit negative → sharper discrimination
- ✅ **Option 4** — Threshold optimized for **F1-score** instead of accuracy
- All original features kept: data augmentation, skill-weighted hybrid score, hard negative mining

In [ ]:
# ============================================================
# CELL 9 — Create MNRL Training Examples
# ============================================================
def create_mnrl_examples(df):
    """
    For MultipleNegativesRankingLoss, we only use POSITIVE pairs.
    Format options:
      - Pair:    InputExample(texts=[resume, matching_jd])
      - Triplet: InputExample(texts=[resume, matching_jd, hard_neg_jd])
    MNRL uses other items in the batch as implicit negatives;
    triplets add explicit hard negatives on top.
    """
    examples = []
    matched   = df[df['Best Match'] == 1].reset_index(drop=True)
    unmatched = df[df['Best Match'] == 0].reset_index(drop=True)

    print(f'📝 Building examples from {len(matched):,} positive pairs...')

    for _, row in tqdm(matched.iterrows(), total=len(matched)):
        resume  = row['Resume']
        pos_jd  = row['Job Description']

        if CONFIG['use_hard_negatives'] and len(unmatched) > 0:
            # Find unmatched JDs with similar skill overlap (hardest negatives)
            skill_diff    = np.abs(unmatched['skill_overlap'] - row['skill_overlap'])
            candidate_idx = skill_diff.nsmallest(5).index
            if len(candidate_idx) > 0:
                hard_neg_jd = unmatched.loc[np.random.choice(candidate_idx), 'Job Description']
                # Triplet: anchor + positive + explicit hard negative
                examples.append(InputExample(texts=[resume, pos_jd, hard_neg_jd]))
                continue

        # Fallback: plain positive pair
        examples.append(InputExample(texts=[resume, pos_jd]))

    triplets = sum(1 for e in examples if len(e.texts) == 3)
    print(f'✅ {len(examples):,} total examples  ({triplets:,} hard-negative triplets)')
    return examples


train_examples_mnrl = create_mnrl_examples(train_df)

In [ ]:
# ============================================================
# CELL 10 — Train Improved Bi-Encoder
# ============================================================
print(f"🤖 Loading base model: {CONFIG['biencoder_base']}")
bi_model = SentenceTransformer(CONFIG['biencoder_base'])
bi_model.max_seq_length = CONFIG['biencoder_max_seq']

train_loader = DataLoader(
    train_examples_mnrl,
    shuffle=True,
    batch_size=CONFIG['biencoder_batch_size']
)

# ✅ OPTION 1 — MultipleNegativesRankingLoss (replaces CosineSimilarityLoss)
train_loss = losses.MultipleNegativesRankingLoss(bi_model)

# Validation evaluator (uses cosine similarity correlation)
val_examples_eval = [
    InputExample(
        texts=[r['Resume'], r['Job Description']],
        label=float(r['Best Match'])
    )
    for _, r in val_df.iterrows()
]
val_evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples_eval, name='val'
)

print(f"\n🏋️  Training for {CONFIG['biencoder_epochs']} epochs...")
print(f"    Batch size : {CONFIG['biencoder_batch_size']}")
print(f"    LR         : {CONFIG['biencoder_lr']}")
print(f"    Loss       : MultipleNegativesRankingLoss  ← NEW")

bi_model.fit(
    train_objectives=[(train_loader, train_loss)],
    evaluator=val_evaluator,
    epochs=CONFIG['biencoder_epochs'],
    evaluation_steps=500,
    warmup_steps=CONFIG['biencoder_warmup'],
    output_path=CONFIG['biencoder_output'],
    save_best_model=True,
    show_progress_bar=True,
    optimizer_params={'lr': CONFIG['biencoder_lr']}
)

print(f"\n✅ Bi-encoder saved → {CONFIG['biencoder_output']}")

In [ ]:
# ============================================================
# CELL 11 — Evaluate Improved Bi-Encoder
# ============================================================
def evaluate_biencoder(model, df, label='Improved Bi-Encoder'):
    resumes     = df['Resume'].tolist()
    jds         = df['Job Description'].tolist()
    true_labels = np.array(df['Best Match'].tolist())
    skill_scores = np.array(df['skill_overlap'].tolist())

    print(f'🔄 Encoding {len(resumes):,} test samples...')
    r_embs = model.encode(resumes,  batch_size=16, show_progress_bar=True, normalize_embeddings=True)
    j_embs = model.encode(jds,      batch_size=16, show_progress_bar=True, normalize_embeddings=True)

    # Cosine similarity (dot product of L2-normalised vectors)
    base_sims   = np.sum(r_embs * j_embs, axis=1)

    # Skill-weighted hybrid score (same as original pipeline)
    similarities = (
        (1 - CONFIG['skill_weight']) * base_sims +
         CONFIG['skill_weight']      * skill_scores
    )

    # ✅ OPTION 4 — Optimize threshold for F1-score (NOT accuracy)
    best_f1, best_threshold = 0.0, 0.5
    for thresh in np.arange(0.10, 0.90, 0.02):
        preds = (similarities >= thresh).astype(int)
        f = f1_score(true_labels, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_threshold = f, thresh

    preds = (similarities >= best_threshold).astype(int)
    acc   = accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true_labels, preds, average='binary')
    try:    auc = roc_auc_score(true_labels, similarities)
    except: auc = 0.0
    cm = confusion_matrix(true_labels, preds)

    print(f'\n{"="*60}')
    print(f'📊 {label}')
    print(f'{"="*60}')
    print(f'  Accuracy  : {acc*100:.2f}%  {"✅" if acc >= 0.75 else "⚠️"}')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1-Score  : {f1*100:.2f}%')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'  Threshold (F1-optimised): {best_threshold:.3f}')
    print(f'\n  Confusion Matrix:')
    print(f'    TN: {cm[0,0]:4d}  FP: {cm[0,1]:4d}')
    print(f'    FN: {cm[1,0]:4d}  TP: {cm[1,1]:4d}')

    metrics = {
        'model': label,
        'accuracy':          float(acc),
        'precision':         float(prec),
        'recall':            float(rec),
        'f1_score':          float(f1),
        'auc_roc':           float(auc),
        'threshold':         float(best_threshold),
        'skill_weight':      CONFIG['skill_weight'],
        'confusion_matrix':  cm.tolist(),
        'improvements': [
            'MultipleNegativesRankingLoss (Option 1)',
            'Hard negative triplets',
            'F1-optimised threshold (Option 4)',
            'Skill-weighted hybrid similarity'
        ],
        'timestamp': datetime.now().isoformat()
    }
    with open(f"{CONFIG['biencoder_output']}/metrics.json", 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\n✅ Metrics saved → {CONFIG['biencoder_output']}/metrics.json")
    return metrics


# Load the best checkpoint that was saved during training
best_bi_model = SentenceTransformer(CONFIG['biencoder_output'])
bi_metrics    = evaluate_biencoder(best_bi_model, test_df)

---
## 🏆 Cross-Encoder Training (Option 3 — Most Impactful)

Unlike the bi-encoder that encodes each text separately, the cross-encoder reads **both resume + JD together** in one forward pass. This means it sees full contextual relationships:

| Bi-Encoder | Cross-Encoder |
|------------|---------------|
| Encodes texts separately, then compares vectors | Reads both texts together with full cross-attention |
| Fast (vectors can be pre-computed) | Slower (must process pairs at inference time) |
| ~61% accuracy (current model) | Expected +10–15% higher accuracy |

**Base model**: `cross-encoder/ms-marco-MiniLM-L-6-v2` — fine-tuned on the ClarityCareers dataset.

In [ ]:
# ============================================================
# CELL 12 — Create Cross-Encoder Training Examples
# ============================================================
# Cross-encoder uses ALL pairs (matched + unmatched), not just positives

def create_ce_examples(df):
    return [
        InputExample(
            texts=[row['Resume'], row['Job Description']],
            label=float(row['Best Match'])
        )
        for _, row in df.iterrows()
    ]

ce_train = create_ce_examples(train_df)
ce_val   = create_ce_examples(val_df)

print(f'✅ Cross-encoder examples created')
print(f'   Train: {len(ce_train):,}  |  Val: {len(ce_val):,}')

In [ ]:
# ============================================================
# CELL 13 — Train Cross-Encoder
# ============================================================
print(f"🤖 Loading cross-encoder base: {CONFIG['crossencoder_base']}")

# num_labels=1 → sigmoid output, binary cross-entropy loss
ce_model = CrossEncoder(
    CONFIG['crossencoder_base'],
    num_labels=1,
    max_length=CONFIG['crossencoder_max_len']
)

ce_loader    = DataLoader(ce_train, shuffle=True, batch_size=CONFIG['crossencoder_batch_size'])
ce_evaluator = CEBinaryClassificationEvaluator.from_input_examples(ce_val, name='val')

warmup_steps = int(len(ce_loader) * CONFIG['crossencoder_epochs'] * 0.1)

print(f"\n🏋️  Training for {CONFIG['crossencoder_epochs']} epochs...")
print(f"    Batch size : {CONFIG['crossencoder_batch_size']}")
print(f"    LR         : {CONFIG['crossencoder_lr']}")
print(f"    Warmup     : {warmup_steps} steps")
print(f"    Architecture : CrossEncoder (reads both texts together) ← NEW")

ce_model.fit(
    train_dataloader=ce_loader,
    evaluator=ce_evaluator,
    epochs=CONFIG['crossencoder_epochs'],
    warmup_steps=warmup_steps,
    output_path=CONFIG['crossencoder_output'],
    save_best_model=True,
    show_progress_bar=True,
    optimizer_params={'lr': CONFIG['crossencoder_lr']}
)

print(f"\n✅ Cross-encoder saved → {CONFIG['crossencoder_output']}")

In [ ]:
# ============================================================
# CELL 14 — Evaluate Cross-Encoder
# ============================================================
def evaluate_crossencoder(model, df, label='Cross-Encoder'):
    resumes     = df['Resume'].tolist()
    jds         = df['Job Description'].tolist()
    true_labels = np.array(df['Best Match'].tolist())

    print(f'Predicting on {len(resumes):,} test samples...')
    pairs  = [[r, j] for r, j in zip(resumes, jds)]
    scores = model.predict(pairs, show_progress_bar=True)
    scores = np.array(scores)

    # Apply sigmoid if scores are raw logits (outside [0, 1])
    if scores.max() > 1.0 or scores.min() < 0.0:
        scores = expit(scores)

    # ✅ OPTION 4 — Optimize threshold for F1-score (NOT accuracy)
    best_f1, best_threshold = 0.0, 0.5
    for thresh in np.arange(0.10, 0.90, 0.02):
        preds = (scores >= thresh).astype(int)
        f = f1_score(true_labels, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_threshold = f, thresh

    preds = (scores >= best_threshold).astype(int)
    acc   = accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true_labels, preds, average='binary')
    try:    auc = roc_auc_score(true_labels, scores)
    except: auc = 0.0
    cm = confusion_matrix(true_labels, preds)

    print(f'\n{"="*60}')
    print(f'📊 {label}')
    print(f'{"="*60}')
    print(f'  Accuracy  : {acc*100:.2f}%  {"✅" if acc >= 0.75 else "⚠️"}')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1-Score  : {f1*100:.2f}%')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'  Threshold (F1-optimised): {best_threshold:.3f}')
    print(f'\n  Confusion Matrix:')
    print(f'    TN: {cm[0,0]:4d}  FP: {cm[0,1]:4d}')
    print(f'    FN: {cm[1,0]:4d}  TP: {cm[1,1]:4d}')

    metrics = {
        'model': label,
        'accuracy':         float(acc),
        'precision':        float(prec),
        'recall':           float(rec),
        'f1_score':         float(f1),
        'auc_roc':          float(auc),
        'threshold':        float(best_threshold),
        'confusion_matrix': cm.tolist(),
        'improvements': [
            'CrossEncoder architecture reads both texts together (Option 3)',
            'Fine-tuned on full dataset',
            'F1-optimised threshold (Option 4)'
        ],
        'timestamp': datetime.now().isoformat()
    }
    with open(f"{CONFIG['crossencoder_output']}/metrics.json", 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\nMetrics saved -> {CONFIG['crossencoder_output']}/metrics.json")
    return metrics


# Use ce_model directly — it already holds best weights after fit().
# Reloading via CrossEncoder(path) fails in sentence-transformers v3.x because
# it saves in its own format, not the HuggingFace AutoConfig format.
best_ce_model = ce_model
ce_metrics    = evaluate_crossencoder(best_ce_model, test_df)

---
## 📈 Final Comparison

In [ ]:
# ============================================================
# CELL 15 — Compare All Models
# ============================================================
BASELINE_ACC = 0.6167   # Current advanced-model accuracy
BASELINE_F1  = 0.6260   # Current advanced-model F1

print()
print('=' * 70)
print('  MODEL COMPARISON SUMMARY')
print('=' * 70)
print(f'{"Model":<32} {"Accuracy":>10} {"F1-Score":>10} {"AUC-ROC":>10}')
print('-' * 70)
print(f'{"Baseline (advanced-model)":<32} {BASELINE_ACC*100:>9.2f}%  {BASELINE_F1*100:>9.2f}%  {"—":>10}')
print(f'{"Improved Bi-Encoder":<32} {bi_metrics["accuracy"]*100:>9.2f}%  {bi_metrics["f1_score"]*100:>9.2f}%  {bi_metrics["auc_roc"]:>10.4f}')
print(f'{"Cross-Encoder":<32} {ce_metrics["accuracy"]*100:>9.2f}%  {ce_metrics["f1_score"]*100:>9.2f}%  {ce_metrics["auc_roc"]:>10.4f}')
print('=' * 70)

bi_gain = (bi_metrics['accuracy'] - BASELINE_ACC) * 100
ce_gain = (ce_metrics['accuracy'] - BASELINE_ACC) * 100

print(f'\n📈 Accuracy gain over baseline:')
print(f'   Improved Bi-Encoder : {bi_gain:+.2f}%')
print(f'   Cross-Encoder       : {ce_gain:+.2f}%')

best_model = 'Cross-Encoder' if ce_metrics['accuracy'] > bi_metrics['accuracy'] else 'Improved Bi-Encoder'
print(f'\n🏆 Best model this run: {best_model}')

print('\n📝 Next steps:')
print('   1. Download both models (next cell)')
print('   2. Paste into your project:')
print('      models/improved-biencoder/')
print('      models/crossencoder/')
print('   3. Share accuracy numbers — integration will be done for the best one')

---
## 💾 Download Models

In [ ]:
# ============================================================
# CELL 16 — Zip & Download Both Models
# ============================================================
from google.colab import files

print('📦 Zipping improved-biencoder...')
shutil.make_archive('improved-biencoder', 'zip', CONFIG['biencoder_output'])
print('✅ improved-biencoder.zip ready')

print('📦 Zipping crossencoder...')
shutil.make_archive('crossencoder', 'zip', CONFIG['crossencoder_output'])
print('✅ crossencoder.zip ready')

print('\n📥 Starting downloads...')
files.download('improved-biencoder.zip')
files.download('crossencoder.zip')

print('\n✅ Done!')
print('\n📂 Unzip each file and place the folders into the project:')
print('     ClarityCareers/models/improved-biencoder/')
print('     ClarityCareers/models/crossencoder/')
print('\n⚠️  Do NOT replace models/advanced-model/ — keep it as the baseline reference.')